# Loan Approval Prediction System — Source Code
**Submitted By:** Neha Chiluka | B.Tech, CSE | Kakatiya Institute of Technology and Science, Warangal | Roll No: B24CS195L

This notebook implements the full pipeline described in the Algorithm & Dataset submission:
data preprocessing → feature engineering → class balancing (SMOTE) → model training
(Logistic Regression, Random Forest, XGBoost) → hyperparameter tuning → evaluation → model export.

**Dataset:** [Loan Prediction Problem Dataset (Kaggle)](https://www.kaggle.com/datasets/altruistdelhite04/loan-prediction-problem-dataset)

Note: The results in this notebook were generated using the real Kaggle dataset (train.csv, 614 records). If data/loan_data.csv is absent, the notebook automatically generates a synthetic dataset with the same schema so the pipeline can still be demonstrated end-to-end.

In [16]:
import os
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Step 1: Data Collection
Load the dataset. Falls back to a synthetic sample if the real Kaggle CSV isn't present.

In [2]:
DATA_PATH = "data/loan_data.csv"

if not os.path.exists(DATA_PATH):
    import subprocess
    subprocess.run(["python3", "data/generate_sample_data.py"], check=True)

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (614, 13)


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    str    
 1   Gender             601 non-null    str    
 2   Married            611 non-null    str    
 3   Dependents         599 non-null    str    
 4   Education          614 non-null    str    
 5   Self_Employed      582 non-null    str    
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    str    
 12  Loan_Status        614 non-null    str    
dtypes: float64(4), int64(1), str(8)
memory usage: 83.4 KB


In [4]:
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64


## Step 2 & 3: Data Cleaning and Encoding
Impute missing values, then encode categorical variables.

In [5]:
df = df.drop(columns=["Loan_ID"])

# Impute categorical fields with mode
for col in ["Gender", "Married", "Dependents", "Self_Employed", "Loan_Amount_Term", "Credit_History"]:
    df[col] = df[col].fillna(df[col].mode()[0])

# Impute LoanAmount with median (robust to outliers)
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())

# Encode Dependents ("3+" -> 3)
df["Dependents"] = df["Dependents"].replace("3+", 3).astype(int)

# Encode target
df["Loan_Status"] = df["Loan_Status"].map({"Y": 1, "N": 0})

# Label-encode binary categoricals
binary_cols = ["Gender", "Married", "Education", "Self_Employed"]
label_encoders = {}
for col in binary_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# One-hot encode Property_Area
df = pd.get_dummies(df, columns=["Property_Area"], drop_first=True)

print("Missing values remaining:", df.isnull().sum().sum())
df.head()

Missing values remaining: 0


,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Property_Area_Semiurban,Property_Area_Urban
0,1,0,0,0,0,5849,0.0,128.0,360.0,1.0,1,False,True
1,1,1,1,0,0,4583,1508.0,128.0,360.0,1.0,0,False,False
2,1,1,0,0,1,3000,0.0,66.0,360.0,1.0,1,False,True
3,1,1,0,1,0,2583,2358.0,120.0,360.0,1.0,1,False,True
4,1,0,0,0,0,6000,0.0,141.0,360.0,1.0,1,False,True


## Step 4: Feature Engineering
Derive Total Income, Debt-to-Income Ratio, and a log-transformed Loan Amount.

In [6]:
df["TotalIncome"] = df["ApplicantIncome"] + df["CoapplicantIncome"]
df["DebtToIncomeRatio"] = df["LoanAmount"] / (df["TotalIncome"] / 1000 + 1e-3)
df["LoanAmount_log"] = np.log1p(df["LoanAmount"])
df["TotalIncome_log"] = np.log1p(df["TotalIncome"])

df[["ApplicantIncome", "CoapplicantIncome", "TotalIncome",
    "LoanAmount", "DebtToIncomeRatio", "LoanAmount_log"]].describe()

,ApplicantIncome,CoapplicantIncome,TotalIncome,LoanAmount,DebtToIncomeRatio,LoanAmount_log
count,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000
mean,5403.459283,1621.245798,7024.705081,145.752443,23.871759,4.866091
std,6109.041673,2926.248369,6458.663872,84.107233,8.703950,0.490811
min,150.000000,0.000000,1442.000000,9.000000,2.522846,2.302585
25%,2877.500000,0.000000,4166.000000,100.250000,19.218411,4.617584
50%,3812.500000,1188.500000,5416.500000,128.000000,24.126079,4.859812
75%,5795.000000,2297.250000,7521.750000,164.750000,28.147329,5.110477
max,81000.000000,41667.000000,81000.000000,700.000000,82.684825,6.552508


## Step 5: Feature Scaling
Split features/target, then standardize numeric columns.

In [7]:
X = df.drop(columns=["Loan_Status"])
y = df["Loan_Status"]

numeric_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term",
                 "TotalIncome", "DebtToIncomeRatio", "LoanAmount_log", "TotalIncome_log"]

scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
X.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area_Semiurban,Property_Area_Urban,TotalIncome,DebtToIncomeRatio,LoanAmount_log,TotalIncome_log
0,1,0,0,0,0,0.072991,-0.554487,-0.211241,0.273231,1.0,False,True,-0.182184,-0.228981,-0.012803,0.008426
1,1,1,1,0,0,-0.134412,-0.038732,-0.211241,0.273231,1.0,False,False,-0.144684,-0.328923,-0.012803,0.082860
2,1,1,0,0,1,-0.393747,-0.554487,-0.948996,0.273231,1.0,False,True,-0.623656,-0.216065,-1.348663,-1.217299
3,1,1,0,1,0,-0.462062,0.251980,-0.306435,0.273231,1.0,False,True,-0.322885,0.047133,-0.143351,-0.301306
4,1,0,0,0,0,0.097728,-0.554487,-0.056551,0.273231,1.0,False,True,-0.158785,-0.043197,0.182981,0.055223


## Step 6: Train-Test Split (Stratified 80/20)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train class balance:\n", y_train.value_counts(normalize=True))

Train shape: (491, 16)  Test shape: (123, 16)
Train class balance:
 Loan_Status
1    0.686354
0    0.313646
Name: proportion, dtype: float64


## Step 7: Handling Class Imbalance with SMOTE
Applied only to the training set to avoid leaking synthetic samples into evaluation.

In [9]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
print("Balanced train class distribution:\n", y_train_bal.value_counts(normalize=True))

Balanced train class distribution:
 Loan_Status
1    0.5
0    0.5
Name: proportion, dtype: float64


## Step 8 & 9: Model Training and Hyperparameter Tuning
Three models are trained and tuned via `GridSearchCV` with 5-fold stratified cross-validation.

In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# --- Logistic Regression ---
lr_grid = {"C": [0.01, 0.1, 1, 10, 100], "penalty": ["l2"], "solver": ["lbfgs"]}
lr_search = GridSearchCV(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                          lr_grid, cv=cv, scoring="f1", n_jobs=-1)
lr_search.fit(X_train_bal, y_train_bal)
print("Best Logistic Regression params:", lr_search.best_params_)

Best Logistic Regression params: {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}


In [11]:
# --- Random Forest ---
rf_grid = {"n_estimators": [100, 200], "max_depth": [5, 10, None], "min_samples_split": [2, 5]}
rf_search = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE),
                          rf_grid, cv=cv, scoring="f1", n_jobs=-1)
rf_search.fit(X_train_bal, y_train_bal)
print("Best Random Forest params:", rf_search.best_params_)

Best Random Forest params: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}


In [12]:
# --- XGBoost ---
xgb_grid = {"n_estimators": [100, 200], "max_depth": [3, 5], "learning_rate": [0.05, 0.1]}
xgb_search = GridSearchCV(XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss"),
                           xgb_grid, cv=cv, scoring="f1", n_jobs=-1)
xgb_search.fit(X_train_bal, y_train_bal)
print("Best XGBoost params:", xgb_search.best_params_)

Best XGBoost params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}


## Step 10: Model Evaluation
Evaluate all three tuned models on the held-out test set.

In [13]:
def evaluate(model, name):
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1-Score": f1_score(y_test, preds),
        "ROC-AUC": roc_auc_score(y_test, proba),
    }
    return metrics, preds

results = []
models = {
    "Logistic Regression": lr_search.best_estimator_,
    "Random Forest": rf_search.best_estimator_,
    "XGBoost": xgb_search.best_estimator_,
}

preds_by_model = {}
for name, model in models.items():
    metrics, preds = evaluate(model, name)
    results.append(metrics)
    preds_by_model[name] = preds

results_df = pd.DataFrame(results).set_index("Model").round(4)
results_df

,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Model,,,,,
Logistic Regression,0.8455,0.8587,0.9294,0.8927,0.8712
Random Forest,0.8455,0.8929,0.8824,0.8876,0.8560
XGBoost,0.8537,0.8764,0.9176,0.8966,0.8553


In [14]:
best_model_name = results_df["F1-Score"].idxmax()
best_model = models[best_model_name]
print(f"Best-performing model: {best_model_name}")
print("\nClassification report:\n")
print(classification_report(y_test, preds_by_model[best_model_name], target_names=["Rejected", "Approved"]))
print("Confusion matrix:\n", confusion_matrix(y_test, preds_by_model[best_model_name]))

Best-performing model: XGBoost

Classification report:

              precision    recall  f1-score   support

    Rejected       0.79      0.71      0.75        38
    Approved       0.88      0.92      0.90        85

    accuracy                           0.85       123
   macro avg       0.84      0.81      0.82       123
weighted avg       0.85      0.85      0.85       123

Confusion matrix:
 [[27 11]
 [ 7 78]]


## Step 11: Deployment — Serialize the Final Model
Save the best model, scaler, and encoders for use in the Streamlit interface (`app.py`).

In [15]:
os.makedirs("models", exist_ok=True)
joblib.dump(best_model, "models/loan_model.pkl")
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(label_encoders, "models/label_encoders.pkl")
joblib.dump(list(X.columns), "models/feature_columns.pkl")

print("Saved: models/loan_model.pkl, scaler.pkl, label_encoders.pkl, feature_columns.pkl")

Saved: models/loan_model.pkl, scaler.pkl, label_encoders.pkl, feature_columns.pkl


## Summary

| Step | Description |
|---|---|
| 1–3 | Data loaded, cleaned (mode/median imputation), categorical variables encoded |
| 4 | Engineered `TotalIncome`, `DebtToIncomeRatio`, log-transformed amount/income |
| 5–6 | Features scaled; stratified 80/20 train-test split |
| 7 | SMOTE applied to training data only, correcting class imbalance |
| 8–9 | Logistic Regression, Random Forest, XGBoost trained and tuned via GridSearchCV |
| 10 | Models compared on Accuracy/Precision/Recall/F1/ROC-AUC; best model selected |
| 11 | Final model, scaler, and encoders serialized for the Streamlit app |

The trained artifacts in `models/` are consumed directly by `app.py` (Streamlit interface) for
real-time predictions.